# Notebook 08 — Self-Supervised Temporal 2D Encoder (Phase 2)
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

## Why this notebook exists

This is **the third leg** of the three-way comparison from `extension_2d_plan.md`:
1. Conventional — APEC (PC1, PC2) BSISO index *(no training, just labels)*.
2. **Supervised contrastive 2D** — pairs defined by BSISO phase + ENSO category. Best result: notebook 07c, phase val **58.3%**, z=**2.53**, 5-fold CV **65.7% ± 4.1%**.
3. **Self-supervised temporal 2D** — pairs defined by *temporal proximity only*, no BSISO/ENSO labels used during training. **This notebook.**

This is the advisor's MJO method, ported to BSISO. The question: does temporal continuity alone discover phase structure?

## Setup (locked decisions from `extension_2d_plan.md`)

| Knob | Value | Source |
|---|---|---|
| Embedding dim | 2 (no L2 norm) | Phase 1 lesson — L2 norm collapses S¹ |
| Loss | raw dot product InfoNCE | Same as Option B (07c) |
| Temperature τ | 0.5 | Best from 07b sweep |
| Weight decay | 1e-4 | Same as Option B (07c) |
| Optimizer | Adam, lr=1e-3, cosine→1e-5 | Same as 04/07/07b/07c |
| Epochs | 50 | Same as prior runs |
| Bandpass | Lanczos lowpass at 25 days, per year | Q1 from plan: yes |
| Edge drop | 25 days from each end of each year | Filter edge effect mitigation |
| Positive pair | anchor d, positive in [d−3, d+3] \ {d}, same year | Q2: ±3 days, locked |
| Negative sampling | implicit in-batch (SimCLR-style) | Standard |
| Same-month restriction | NO | Lee + bandpass already remove seasonal cycle |
| Data | `labels_aligned_mjjas_lee_clean.csv` (notebook 03b) | Bug-free amplitude column |

## What gets produced

- `checkpoints/encoder_2d_lee_ssl_final.pth`
- `checkpoints/training_history_2d_lee_ssl.json`
- `data/processed/X_MJJAS_lee_lp25.npy` — bandpassed input fields (saved for reuse by Phase 3)
- `data/processed/labels_aligned_mjjas_lee_lp25.csv` — labels aligned to bandpassed frames (edge days dropped)
- `results/lee_2d_ssl/`
  - `embeddings.npy` — (N_filt, 2) raw 2D, NOT normalized
  - `training_curves.png` — loss + norm trajectory
  - `embedding_2d_overview.png` — 4-panel: by phase / ENSO / **calendar month** / amplitude
  - `month_clustering_check.png` — boxplot of angles by month (the critical confound check)
  - `linear_probe_results.json`
  - `enso_displacement.png`
  - `phase2_ssl_summary.md` — comparison table + auto-decision

## Decision rule

- **Greenlight Phase 3 (three-way comparison)**: phase val ≥ 40% (well above 12.5% random; ≥ ⅔ of supervised 58.3% would be a strong SSL result). Notebook 09 then runs.
- **Month confound**: if SSL embeddings cluster strongly by calendar month (visible in scatter, ANOVA F > 50 on angle by month), the bandpass wasn't enough — recommend tightening (shorter lowpass cutoff or restricting negatives to same month).
- **Below threshold**: phase val < 25% → temporal continuity alone isn't sufficient under this preprocessing; need to revisit pair definition or bandpass.

## Runtime

Bandpass preprocessing: ~3 min. Training: ~30 min on T4. Total ~35 min.

---
## Before running
1. Notebook 03b should have run (we use the cleaned labels).
2. Notebook 07c should have run (we compare against the supervised 2D baseline).

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy.ndimage import convolve1d

# ── Phase 2 config ───────────────────────────────────────────────────────────
EMBEDDING_DIM   = 2
RUN_TAG         = '2d_lee_ssl'
TEMPERATURE     = 0.5
EPOCHS          = 50
BATCH_SIZE      = 64
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
MAX_DELTA       = 3       # positive-pair window: [d-3, d+3] \ {d}
LOWPASS_PERIOD  = 25      # days; passes signals slower than 25 days
LOWPASS_HALF_W  = 25      # half-window for Lanczos filter (length = 51)
EDGE_DROP_DAYS  = LOWPASS_HALF_W   # drop edge days affected by filter
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
RESULTS_DIR    = f'{PROJECT_DIR}/results/lee_2d_ssl'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

X_FILE      = 'X_MJJAS_lee.npy'
LABELS_FILE = 'labels_aligned_mjjas_lee_clean.csv'   # ← cleaned by 03b
X_BP_FILE   = 'X_MJJAS_lee_lp25.npy'
LABELS_BP_FILE = 'labels_aligned_mjjas_lee_lp25.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:           {device}')
print(f'Embedding dim:    {EMBEDDING_DIM}  (no L2 normalization)')
print(f'Loss:             raw dot product InfoNCE  (τ={TEMPERATURE})')
print(f'Bandpass:         Lanczos lowpass @ {LOWPASS_PERIOD} d (half_w={LOWPASS_HALF_W}, drop ±{EDGE_DROP_DAYS} d/year)')
print(f'Positive pair:    [d−{MAX_DELTA}, d+{MAX_DELTA}] \\ {{d}}, same year')
print(f'Run tag:          {RUN_TAG}')
print(f'Results dir:      results/lee_2d_ssl/')

## Cell 2 — Load Data + Cleaned Labels

If the cleaned labels file (`labels_aligned_mjjas_lee_clean.csv`) doesn't exist yet, fall back to the original. The cleaned file just has NaN in place of bad amplitude values; everything else is identical, so SSL training and probing on phase/ENSO are unaffected.

In [ ]:
X = np.load(f'{PROCESSED_DIR}/{X_FILE}')
print(f'X (raw Lee):  {X.shape}')

clean_path = f'{PROCESSED_DIR}/{LABELS_FILE}'
if os.path.exists(clean_path):
    labels = pd.read_csv(clean_path, parse_dates=['date'])
    print(f'Labels (cleaned): {LABELS_FILE}')
else:
    print(f'WARNING: cleaned labels not found, falling back to original. Run notebook 03b first.')
    labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned_mjjas_lee.csv', parse_dates=['date'])

print(f'Labels rows: {len(labels)}')
print(f'Columns:     {list(labels.columns)}')
assert len(labels) == len(X), f'Mismatch: X={len(X)} vs labels={len(labels)}'

print(f'\nDate range: {labels["date"].min().date()} to {labels["date"].max().date()}')
print(f'\nMonth distribution:')
print(labels['date'].dt.month.value_counts().sort_index().to_string())

## Cell 3 — Lanczos Lowpass Bandpass (per year)

**Filter design:**
- Lanczos lowpass at 25 days (passes intraseasonal periods, removes synoptic <25 days).
- Highpass side (>90 days) is approximately handled by the Lee preprocessing's 120-day running mean removal, so we don't add a separate highpass.
- Half-window 25 days → filter length 51 < MJJAS season length 153 days.
- Apply per-year (each MJJAS season independent — we don't have continuous Oct–Apr data).
- Drop first/last 25 days of each year to avoid filter edge effects.

Result: `~103 days × 43 years ≈ 4,400` valid samples (down from 6,579).

In [ ]:
def lanczos_lowpass_weights(cutoff_days, half_window):
    """Lanczos-windowed sinc lowpass filter.
    Passes signals with periods longer than `cutoff_days` (frequencies < 1/cutoff_days).
    """
    k = np.arange(-half_window, half_window + 1)
    fc = 1.0 / cutoff_days
    # Sinc lowpass kernel
    h = np.where(k == 0, 2*fc, np.sin(2*np.pi*fc*k) / (np.pi*k))
    # Lanczos window: sinc(k / half_window)
    sigma = np.where(k == 0, 1.0, np.sin(np.pi*k/half_window) / (np.pi*k/half_window))
    w = h * sigma
    w /= w.sum()   # normalize so DC gain = 1
    return w

weights = lanczos_lowpass_weights(LOWPASS_PERIOD, LOWPASS_HALF_W)
print(f'Filter length: {len(weights)}  (half_window={LOWPASS_HALF_W})')
print(f'Sum of weights: {weights.sum():.6f}  (should be ~1.0)')

# Quick filter response visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].stem(np.arange(-LOWPASS_HALF_W, LOWPASS_HALF_W+1), weights)
axes[0].set_xlabel('k (days)'); axes[0].set_ylabel('weight')
axes[0].set_title(f'Lanczos lowpass impulse response (cutoff {LOWPASS_PERIOD} d)', fontweight='bold')
axes[0].grid(alpha=0.3)
# Frequency response
freqs = np.linspace(0, 0.5, 500)
response = np.abs(np.array([np.sum(weights * np.exp(-2j*np.pi*f*np.arange(-LOWPASS_HALF_W, LOWPASS_HALF_W+1))) for f in freqs]))
periods = 1.0 / np.where(freqs > 0, freqs, np.inf)
axes[1].plot(periods, response, linewidth=2)
axes[1].axvline(LOWPASS_PERIOD, color='red', linestyle='--', label=f'cutoff {LOWPASS_PERIOD} d')
axes[1].axvline(90, color='gray', linestyle=':', label='BSISO upper edge 90 d (Lee handles)')
axes[1].set_xscale('log'); axes[1].set_xlabel('period (days)'); axes[1].set_ylabel('|H(f)|')
axes[1].set_title('Frequency response', fontweight='bold')
axes[1].set_xlim(2, 200); axes[1].legend(); axes[1].grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

# Apply per-year
years = labels['date'].dt.year.values
X_filt = np.empty_like(X)
mask_keep = np.zeros(len(X), dtype=bool)
for y in np.unique(years):
    idx_year = np.where(years == y)[0]
    # Convolve along axis 0 (time), broadcast over (channel, lat, lon)
    X_filt[idx_year] = convolve1d(X[idx_year], weights, axis=0, mode='reflect')
    if len(idx_year) > 2 * EDGE_DROP_DAYS:
        mask_keep[idx_year[EDGE_DROP_DAYS : -EDGE_DROP_DAYS]] = True

print(f'\nBefore drop: {len(X)} samples')
print(f'After drop:  {int(mask_keep.sum())} samples  ({100*mask_keep.sum()/len(X):.1f}%)')
print(f'Per-year retained: {int(mask_keep.sum())/len(np.unique(years)):.1f} days/year (was {len(X)/len(np.unique(years)):.1f})')

# Subset and save
X_bp = X_filt[mask_keep]
labels_bp = labels.loc[mask_keep].reset_index(drop=True)

np.save(f'{PROCESSED_DIR}/{X_BP_FILE}', X_bp)
labels_bp.to_csv(f'{PROCESSED_DIR}/{LABELS_BP_FILE}', index=False)
print(f'\nSaved: {X_BP_FILE}  shape {X_bp.shape}')
print(f'Saved: {LABELS_BP_FILE}  rows {len(labels_bp)}')

# Sanity: variance of bandpassed vs raw
print(f'\nVariance comparison (channel-mean over space):')
for ch, name in enumerate(['u850', 'v850', 'OLR']):
    var_raw = X[mask_keep, ch].var()
    var_bp  = X_bp[:, ch].var()
    print(f'  {name}: raw var={var_raw:.4f}, bandpassed var={var_bp:.4f}  (ratio={var_bp/var_raw:.2%})')

## Cell 4 — Year-Based Train/Val Split (on Bandpassed Data)

Same logic as 04/07/07b/07c — every 5th year held out. Indices are local to the bandpassed subset (`labels_bp` has been reset_index).

In [ ]:
all_years = sorted(labels_bp['date'].dt.year.unique())
val_years = all_years[::5]
train_years = [y for y in all_years if y not in val_years]

year_col = labels_bp['date'].dt.year
train_idx = labels_bp.index[year_col.isin(train_years)].values
val_idx   = labels_bp.index[year_col.isin(val_years)].values

print(f'Val years ({len(val_years)}): {val_years}')
print(f'Train: {len(train_idx)} samples ({100*len(train_idx)/len(labels_bp):.1f}%)')
print(f'Val:   {len(val_idx)} samples ({100*len(val_idx)/len(labels_bp):.1f}%)')

val_enso = labels_bp.loc[val_idx, 'enso_category'].value_counts()
print(f'\nENSO distribution in val set:')
print(val_enso)

## Cell 5 — SSL Temporal Pair Sampler + Dataset

Key difference from 07/07b/07c: pairs are defined by **temporal proximity only**, no BSISO/ENSO labels.

- Anchor d: any train-set day.
- Positive: random day in [d−3, d+3] \ {d}, same year. (Both anchor and positive must be in the train set, else fall back.)
- Negatives: implicit, via in-batch off-diagonals (SimCLR-style).

No `phase_enso_index` is built; the model never sees BSISO phase or ENSO during training.

In [ ]:
class TemporalPairSampler:
    def __init__(self, labels_df, allowed_indices, max_delta=3):
        self.labels = labels_df
        self.allowed_set = set(int(i) for i in allowed_indices)
        self.max_delta = max_delta
        # Lookup: date → index (only over allowed_indices)
        sub = labels_df.loc[list(self.allowed_set)]
        self.date_to_idx = dict(zip(sub['date'].values, sub.index.values))

    def sample_positive_pair(self, anchor_idx):
        """Returns (anchor_idx, positive_idx) where positive ∈ [d-Δ, d+Δ] \\ {d}, same year.
        Falls back to (anchor, anchor) if no neighbour exists in the allowed set."""
        anchor_date = self.labels.loc[anchor_idx, 'date']
        anchor_year = anchor_date.year
        candidates = []
        for delta in range(-self.max_delta, self.max_delta + 1):
            if delta == 0:
                continue
            target_date = anchor_date + pd.Timedelta(days=delta)
            if target_date.year != anchor_year:
                continue
            target_np = np.datetime64(target_date)
            if target_np in self.date_to_idx:
                candidates.append(self.date_to_idx[target_np])
        if not candidates:
            return anchor_idx, anchor_idx
        return anchor_idx, int(np.random.choice(candidates))


class SSLTemporalDataset(Dataset):
    def __init__(self, X_data, labels_df, train_indices, max_delta=3, mode='train', n_val_pairs=1000):
        self.X = X_data
        self.labels = labels_df
        self.train_indices = np.asarray(train_indices)
        self.sampler = TemporalPairSampler(labels_df, train_indices, max_delta)
        self.mode = mode
        if mode == 'val':
            rng = np.random.default_rng(42)
            self.val_pairs = []
            for idx in rng.choice(self.train_indices, size=min(n_val_pairs * 2, len(self.train_indices)), replace=False):
                a, b = self.sampler.sample_positive_pair(int(idx))
                if a != b:
                    self.val_pairs.append((a, b))
                if len(self.val_pairs) >= n_val_pairs:
                    break

    def __len__(self):
        return len(self.train_indices) if self.mode == 'train' else len(self.val_pairs)

    def __getitem__(self, idx):
        if self.mode == 'train':
            anchor_idx = int(self.train_indices[idx])
            a, b = self.sampler.sample_positive_pair(anchor_idx)
        else:
            a, b = self.val_pairs[idx]
        return torch.from_numpy(self.X[a]).float(), torch.from_numpy(self.X[b]).float()


train_dataset = SSLTemporalDataset(X_bp, labels_bp, train_idx, max_delta=MAX_DELTA, mode='train')
val_dataset   = SSLTemporalDataset(X_bp, labels_bp, val_idx,   max_delta=MAX_DELTA, mode='val')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train dataset: {len(train_dataset)} items  →  {len(train_loader)} batches/epoch')
print(f'Val dataset:   {len(val_dataset)} pairs   →  {len(val_loader)} batches')

# Sanity: inspect a few sampled pairs
print(f'\nSample positive pairs (anchor → positive):')
for i in range(5):
    a, b = train_dataset.sampler.sample_positive_pair(int(train_idx[i]))
    da = labels_bp.loc[a, 'date']; db = labels_bp.loc[b, 'date']
    print(f'  {da.date()} (P{labels_bp.loc[a, "bsiso_phase"]}) → {db.date()} (P{labels_bp.loc[b, "bsiso_phase"]})  Δ={(db-da).days}d')

## Cell 6 — Encoder + Loss (same as 07c — no L2 norm, raw dot product InfoNCE)

In [ ]:
class CNNEncoderNoL2(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc          = nn.Linear(128, embedding_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).view(x.size(0), -1)
        z = self.fc(x)
        return z   # NO L2 normalization


def InfoNCE_loss_raw(z_A, z_B, temperature):
    sim_matrix = torch.matmul(z_A, z_B.T) / temperature
    labels_ = torch.arange(z_A.size(0), device=z_A.device)
    return F.cross_entropy(sim_matrix, labels_)


encoder = CNNEncoderNoL2(embedding_dim=EMBEDDING_DIM).to(device)
optimizer = optim.Adam(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f'Parameters: {sum(p.numel() for p in encoder.parameters()):,}')
print(f'Training:   {EPOCHS} epochs, batch {BATCH_SIZE}, τ={TEMPERATURE}, weight_decay={WEIGHT_DECAY}')

## Cell 7 — Training Loop with Norm Trajectory

In [ ]:
from tqdm.notebook import tqdm

history = {'train_loss': [], 'val_loss': [], 'epoch_time': [],
           'mean_norm': [], 'std_norm': [], 'max_norm': []}

for epoch in range(EPOCHS):
    t0 = time.time()
    encoder.train()
    train_loss = 0.0
    epoch_norms = []
    pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{EPOCHS}', leave=False)
    for fA, fB in pbar:
        fA = fA.to(device, non_blocking=True)
        fB = fB.to(device, non_blocking=True)
        zA = encoder(fA); zB = encoder(fB)
        with torch.no_grad():
            epoch_norms.extend(zA.norm(dim=1).cpu().numpy().tolist())
            epoch_norms.extend(zB.norm(dim=1).cpu().numpy().tolist())
        loss = InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    train_loss /= len(train_loader)

    encoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for fA, fB in val_loader:
            fA = fA.to(device, non_blocking=True); fB = fB.to(device, non_blocking=True)
            zA = encoder(fA); zB = encoder(fB)
            val_loss += InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE).item()
    val_loss /= len(val_loader)
    scheduler.step()

    en = np.array(epoch_norms)
    et = time.time() - t0
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['epoch_time'].append(et)
    history['mean_norm'].append(float(en.mean())); history['std_norm'].append(float(en.std()))
    history['max_norm'].append(float(en.max()))
    print(f'ep {epoch+1:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'norm μ={en.mean():.3f} σ={en.std():.3f} max={en.max():.2f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if (epoch + 1) % 10 == 0:
        torch.save({'epoch': epoch+1, 'model_state_dict': encoder.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss, 'val_loss': val_loss},
                   f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_epoch_{epoch+1}.pth')

torch.save(encoder.state_dict(), f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_final.pth')
with open(f'{CHECKPOINT_DIR}/training_history_{RUN_TAG}.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete. Total time: {sum(history["epoch_time"])/60:.1f} min')
print(f'Final mean norm: {history["mean_norm"][-1]:.3f}  Final max norm: {history["max_norm"][-1]:.2f}')

## Cell 8 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
axes[0].axhline(np.log(64), color='gray', linestyle='--', alpha=0.5, label='log(64)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title(f'SSL Training Curves (τ={TEMPERATURE})', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['mean_norm'], label='Mean norm', linewidth=2)
axes[1].fill_between(range(len(history['mean_norm'])),
                     np.array(history['mean_norm']) - np.array(history['std_norm']),
                     np.array(history['mean_norm']) + np.array(history['std_norm']),
                     alpha=0.3, label='±1σ')
axes[1].plot(history['max_norm'], label='Max norm', color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Embedding norm')
axes[1].set_title('Norm Trajectory', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['epoch_time'], color='green', linewidth=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Time (s)')
axes[2].set_title('Epoch Time', fontweight='bold'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: results/lee_2d_ssl/training_curves.png')

if max(history['max_norm']) > 100:
    print('\n⚠️  Norm explosion warning. Consider increasing weight_decay.')
else:
    print(f'\n✓ Norms stable (max ever = {max(history["max_norm"]):.2f})')

## Cell 9 — Extract Embeddings

In [ ]:
encoder.eval()
embeddings_2d = np.zeros((len(X_bp), EMBEDDING_DIM), dtype=np.float32)
with torch.no_grad():
    for start in range(0, len(X_bp), 128):
        end = min(start + 128, len(X_bp))
        batch = torch.from_numpy(X_bp[start:end]).float().to(device)
        embeddings_2d[start:end] = encoder(batch).cpu().numpy()

np.save(f'{RESULTS_DIR}/embeddings.npy', embeddings_2d)

norms = np.linalg.norm(embeddings_2d, axis=1)
angles = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
print(f'Embeddings shape: {embeddings_2d.shape}')
print(f'Norm:  min={norms.min():.3f} max={norms.max():.3f} mean={norms.mean():.3f} std={norms.std():.3f}')
print(f'Angle: min={angles.min():.3f} max={angles.max():.3f} spread={angles.max()-angles.min():.3f} rad')

## Cell 10 — 4-Panel Scatter (val) — by Phase, ENSO, **Calendar Month**, Amplitude

**Critical new diagnostic:** the third panel colors by calendar month (May/Jun/Jul/Aug/Sep). If embeddings cluster by month, the seasonal cycle confound wasn't fully removed by Lee preprocessing + bandpass — meaning the SSL model learned "this is July" rather than "this is BSISO phase X". That would invalidate the SSL result.

In [ ]:
Z_val = embeddings_2d[val_idx]
lv = labels_bp.loc[val_idx]

phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
enso_palette = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker  = {'El Nino': '^',       'Neutral': 'o',        'La Nina': 's'}
month_colors = plt.cm.viridis(np.linspace(0, 1, 5))
month_names  = ['May', 'Jun', 'Jul', 'Aug', 'Sep']

rng_lo = min(Z_val.min(), -1.0); rng_hi = max(Z_val.max(), 1.0)
pad = 0.1 * (rng_hi - rng_lo); rng_lo -= pad; rng_hi += pad

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# (a) by BSISO phase
ax = axes[0]
for ph in range(1, 9):
    m = lv['bsiso_phase'] == ph
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[phase_colors[ph-1]], s=18, alpha=0.7, label=f'P{ph}')
ax.set_title('By BSISO Phase\n(SSL never saw phase labels)', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.2)

# (b) by ENSO
ax = axes[1]
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = lv['enso_category'] == cat
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=enso_palette[cat], marker=enso_marker[cat], s=18, alpha=0.5, label=cat)
ax.set_title('By ENSO\n(SSL never saw ENSO labels)', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(); ax.grid(alpha=0.2)

# (c) by CALENDAR MONTH — the critical confound check
ax = axes[2]
months = lv['date'].dt.month.values
for mi, mname in enumerate(month_names, start=5):
    m = months == mi
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[month_colors[mi-5]], s=18, alpha=0.7, label=mname)
ax.set_title('By Calendar Month\n(should NOT cluster by month)', fontweight='bold', color='red')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(); ax.grid(alpha=0.2)

# (d) by BSISO amplitude (continuous)
ax = axes[3]
ampl_val = lv['bsiso_amplitude'].values
valid_amp = ~np.isnan(ampl_val)
sc = ax.scatter(Z_val[valid_amp, 0], Z_val[valid_amp, 1], c=ampl_val[valid_amp],
                cmap='viridis', s=18, alpha=0.7)
ax.set_title('By BSISO Amplitude', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
plt.colorbar(sc, ax=ax, label='BSISO amplitude'); ax.grid(alpha=0.2)

plt.suptitle(f'SSL Temporal 2D Encoder (Phase 2) — Val Embeddings  (n={len(Z_val)})',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/embedding_2d_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_ssl/embedding_2d_overview.png')

## Cell 11 — Month-Clustering Confound Check (the formal test)

Quantitative version of the by-month scatter. Tests whether embedding angle θ depends systematically on calendar month using one-way ANOVA. Strong F-statistic → seasonal cycle confound dominates the SSL signal.

In [ ]:
from scipy.stats import f_oneway

angles_all = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
months_all = labels_bp['date'].dt.month.values
radii_all  = np.linalg.norm(embeddings_2d, axis=1)

angles_by_month = [angles_all[months_all == m] for m in [5, 6, 7, 8, 9]]
radii_by_month  = [radii_all[months_all == m]  for m in [5, 6, 7, 8, 9]]

f_a, p_a = f_oneway(*angles_by_month)
f_r, p_r = f_oneway(*radii_by_month)
print(f'Angle by month  ANOVA: F = {f_a:.2f}  p = {p_a:.2e}')
print(f'Radius by month ANOVA: F = {f_r:.2f}  p = {p_r:.2e}')

# Boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
bp = ax.boxplot(angles_by_month, positions=[5,6,7,8,9], widths=0.6, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], plt.cm.viridis(np.linspace(0, 1, 5))):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticks([5,6,7,8,9]); ax.set_xticklabels(['May','Jun','Jul','Aug','Sep'])
ax.set_xlabel('Calendar month'); ax.set_ylabel('Embedding angle θ (rad)')
ax.set_title(f'Angle by Month  ANOVA F = {f_a:.2f}  (p = {p_a:.2e})', fontweight='bold')
ax.grid(alpha=0.3)

ax = axes[1]
bp = ax.boxplot(radii_by_month, positions=[5,6,7,8,9], widths=0.6, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], plt.cm.viridis(np.linspace(0, 1, 5))):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xticks([5,6,7,8,9]); ax.set_xticklabels(['May','Jun','Jul','Aug','Sep'])
ax.set_xlabel('Calendar month'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius by Month  ANOVA F = {f_r:.2f}  (p = {p_r:.2e})', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Month-Clustering Confound Check (SSL)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/month_clustering_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_ssl/month_clustering_check.png')

# Verdict
if f_a > 50:
    print(f'\n⚠️  Strong month confound on angle (F={f_a:.0f}). Bandpass + Lee was insufficient.')
    print('   Recommend: shorter lowpass cutoff, or restrict negatives to same month.')
elif f_a > 10:
    print(f'\n△ Moderate month dependence on angle (F={f_a:.1f}). Likely intraseasonal-amplitude modulation across MJJAS, may be acceptable.')
else:
    print(f'\n✓ Month dependence on angle is weak (F={f_a:.1f}). Confound under control.')

## Cell 12 — Linear Probes (BSISO Phase + ENSO Balanced)

Strict test of the SSL model: did temporal continuity alone discover BSISO phase structure? Compare to:
- Random baselines: 12.5% (phase, 8 classes), 33.3% (ENSO bal-acc, 3 classes).
- 2D supervised Option B (07c): phase val 58.3%, 5-fold CV 65.7%; ENSO bal-acc 34.6%.
- 64D supervised: phase val 67.7%, 5-fold CV 68.6%.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GroupKFold

year_groups = labels_bp['date'].dt.year.values
Z_train = embeddings_2d[train_idx]
Z_val_  = embeddings_2d[val_idx]
gkf = GroupKFold(n_splits=5)

probe_results = {}

# BSISO phase
y_tr = labels_bp.loc[train_idx, 'bsiso_phase'].values
y_va = labels_bp.loc[val_idx,   'bsiso_phase'].values
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(Z_train, y_tr)
phase_val = float(accuracy_score(y_va, clf.predict(Z_val_)))
cv_phase = cross_val_score(clf, embeddings_2d, labels_bp['bsiso_phase'].values,
                            cv=gkf, groups=year_groups, scoring='accuracy', n_jobs=-1)

# ENSO balanced
y_tr = labels_bp.loc[train_idx, 'enso_category'].values
y_va = labels_bp.loc[val_idx,   'enso_category'].values
clf_b = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
clf_b.fit(Z_train, y_tr)
enso_bal_val = float(balanced_accuracy_score(y_va, clf_b.predict(Z_val_)))
cv_enso = cross_val_score(clf_b, embeddings_2d, labels_bp['enso_category'].values,
                           cv=gkf, groups=year_groups, scoring='balanced_accuracy', n_jobs=-1)

probe_results['BSISO Phase'] = {
    'val_acc': phase_val, 'cv_mean': float(cv_phase.mean()), 'cv_std': float(cv_phase.std())
}
probe_results['ENSO Category (balanced)'] = {
    'val_bal_acc': enso_bal_val, 'cv_mean': float(cv_enso.mean()), 'cv_std': float(cv_enso.std())
}

print('=' * 70)
print('SSL TEMPORAL — LINEAR PROBE RESULTS (no labels seen during training)')
print('=' * 70)
print(f'BSISO phase val acc:  {phase_val*100:.1f}%   (random 12.5%)')
print(f'BSISO phase 5-fold CV: {cv_phase.mean()*100:.1f}% ± {cv_phase.std()*100:.1f}%')
print(f'ENSO  bal-acc val:    {enso_bal_val*100:.1f}%   (random 33.3%)')
print(f'ENSO  bal-acc 5-fold: {cv_enso.mean()*100:.1f}% ± {cv_enso.std()*100:.1f}%')
print(f'\nReference values:')
print(f'  64D supervised:    phase 67.7%, CV 68.6% ± 1.0%')
print(f'  2D supervised (B): phase 58.3%, CV 65.7% ± 4.1%, ENSO bal 34.6%')

print(f'\nVal classification report (BSISO phase):')
print(classification_report(y_va, clf.predict(Z_val_), zero_division=0))

with open(f'{RESULTS_DIR}/linear_probe_results.json', 'w') as f:
    json.dump(probe_results, f, indent=2)
print(f'\nSaved: results/lee_2d_ssl/linear_probe_results.json')

## Cell 13 — ENSO Displacement Z-Score

Does ENSO modulation emerge in the SSL embedding space *without* the model being told ENSO categories? If z > 2, that's a strong result — temporal continuity alone discovers ENSO-modulated BSISO structure.

In [ ]:
phases = range(1, 9)
disp_mag = []
for ph in phases:
    mEN = (labels_bp['bsiso_phase'] == ph) & (labels_bp['enso_category'] == 'El Nino')
    mLN = (labels_bp['bsiso_phase'] == ph) & (labels_bp['enso_category'] == 'La Nina')
    if mEN.sum() < 3 or mLN.sum() < 3:
        disp_mag.append(np.nan); continue
    cEN = embeddings_2d[mEN].mean(axis=0); cLN = embeddings_2d[mLN].mean(axis=0)
    disp_mag.append(np.linalg.norm(cEN - cLN))

rng = np.random.default_rng(42)
baseline_mag = []
for _ in range(100):
    shuf = labels_bp['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
    mtrl = []
    for ph in phases:
        mph = (labels_bp['bsiso_phase'] == ph).values
        mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        mtrl.append(np.linalg.norm(embeddings_2d[mEN].mean(axis=0) - embeddings_2d[mLN].mean(axis=0)))
    if mtrl: baseline_mag.append(np.mean(mtrl))

bmu = float(np.mean(baseline_mag)); bsd = float(np.std(baseline_mag))
obs_mu = float(np.nanmean(disp_mag))
z_score_ssl = float((obs_mu - bmu) / (bsd + 1e-8))

print(f'EN−LN displacement summary (SSL):')
print(f'  Observed mean: {obs_mu:.4f}')
print(f'  Null baseline: {bmu:.4f} ± {bsd:.4f}')
print(f'  Z-score:       {z_score_ssl:.2f}   (64D supervised: 3.83;  2D supervised: 2.53)')

fig, ax = plt.subplots(figsize=(8, 5))
valid_phases = [p for p, m in zip(phases, disp_mag) if not np.isnan(m)]
valid_mags = [m for m in disp_mag if not np.isnan(m)]
ax.bar(valid_phases, valid_mags, color='steelblue', alpha=0.8)
ax.axhline(bmu, color='red', linestyle='--', linewidth=1.5, label=f'Null mean ({bmu:.3f})')
ax.axhline(bmu + 2*bsd, color='red', linestyle=':', linewidth=1, label='Null +2σ')
ax.axhline(obs_mu, color='steelblue', linewidth=2, label=f'Observed mean ({obs_mu:.3f})')
ax.set_xticks(range(1, 9))
ax.set_xlabel('BSISO Phase'); ax.set_ylabel('||EN−LN||')
ax.set_title(f'SSL ENSO Displacement, z = {z_score_ssl:.2f}  (no ENSO labels in training)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enso_displacement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_ssl/enso_displacement.png')

## Cell 14 — Phase 2 Summary: Comparison Table + Auto-Decision

Writes the headline three-way comparison for the writeup. The conventional (PC1, PC2) leg will be added in notebook 09.

In [ ]:
import time as _t

# Hardcoded baselines from prior runs
BASELINES = {
    '64D supervised (Lee MJJAS, notebook 04)': {'phase_val': 0.677, 'phase_cv': '68.6% ± 1.0%', 'enso_bal': 'n/a', 'z': 3.83, 'sees_labels': 'phase + ENSO'},
    '2D supervised Option B (notebook 07c)':    {'phase_val': 0.583, 'phase_cv': '65.7% ± 4.1%', 'enso_bal': '34.6%', 'z': 2.53, 'sees_labels': 'phase + ENSO'},
}

rows = []
for name, b in BASELINES.items():
    rows.append([name, f"{b['phase_val']*100:.1f}%", b['phase_cv'], str(b['enso_bal']), f"{b['z']:.2f}", b['sees_labels']])
rows.append(['**2D SSL temporal (this notebook)**',
             f"{phase_val*100:.1f}%",
             f"{cv_phase.mean()*100:.1f}% ± {cv_phase.std()*100:.1f}%",
             f"{enso_bal_val*100:.1f}%",
             f"{z_score_ssl:.2f}",
             '**none (temporal only)**'])
df_comp = pd.DataFrame(rows, columns=['Configuration', 'BSISO phase val', 'BSISO 5-fold CV', 'ENSO bal-acc', 'z-score', 'Labels seen during training'])

print('=' * 130)
print('PHASE 2 SUMMARY — Three-Way Comparison (preliminary; conventional leg added in notebook 09)')
print('=' * 130)
print(df_comp.to_string(index=False))

# Auto-decision
phase_pass    = phase_val >= 0.40                # ≥ 40% = strong SSL result vs 12.5% random
z_pass        = z_score_ssl >= 2.0               # ≥ 2 = significant ENSO emergence
month_confound = f_a > 50                        # ANOVA F-stat threshold
no_signal     = phase_val < 0.25                 # below 2× random

if month_confound:
    decision = 'MONTH_CONFOUND'
    decision_text = (f"**Month-clustering confound detected** (angle ANOVA F={f_a:.1f}). "
                     f"Lee preprocessing + Lanczos lowpass at 25 d was insufficient. "
                     f"Recommend: shorter lowpass cutoff (e.g. 20 d), or restrict negatives to same calendar month.")
elif phase_pass and z_pass:
    decision = 'GREENLIGHT_PHASE_3'
    decision_text = (f"**SSL temporal continuity discovered BSISO phase structure without labels.** "
                     f"Phase val {phase_val*100:.1f}% (≥ 40% threshold), z-score {z_score_ssl:.2f} (≥ 2.0). "
                     f"Compared to 2D supervised (phase val 58.3%, z 2.53) — SSL captures "
                     f"{phase_val/0.583*100:.0f}% of supervised phase performance and "
                     f"{z_score_ssl/2.53*100:.0f}% of supervised ENSO modulation. "
                     f"→ **Greenlight Phase 3** (notebook 09 — three-way comparison with conventional baseline + lag probes).")
elif phase_pass:
    decision = 'PHASE_OK_ENSO_WEAK'
    decision_text = (f"**SSL captured phase structure but ENSO modulation is weak** (phase val {phase_val*100:.1f}%, z {z_score_ssl:.2f}). "
                     f"Phase 3 still meaningful — document that SSL underplays ENSO emergence vs supervised.")
elif no_signal:
    decision = 'NO_SIGNAL'
    decision_text = (f"**Temporal continuity alone is insufficient** (phase val {phase_val*100:.1f}% ≈ random). "
                     f"Pair definition or preprocessing needs revision. Possible fixes: longer positive window (±7 d), "
                     f"different bandpass, alternative loss.")
else:
    decision = 'PARTIAL'
    decision_text = (f"**Partial SSL signal** (phase val {phase_val*100:.1f}%, z {z_score_ssl:.2f}). "
                     f"Discuss with advisor before Phase 3.")

print('\n' + '=' * 70)
print(f'DECISION: {decision}')
print('=' * 70)
print(decision_text)

# Write summary md
summary = f"""# Phase 2 — SSL Temporal 2D Encoder Summary

**Auto-generated by notebook 08.**  
**Date:** {_t.strftime('%Y-%m-%d')}  
**Model:** `encoder_{RUN_TAG}_final.pth`, no L2 norm, τ={TEMPERATURE}, weight_decay={WEIGHT_DECAY}  
**Pair definition:** anchor d, positive in [d−{MAX_DELTA}, d+{MAX_DELTA}] \\ {{d}}, same year. **No BSISO/ENSO labels.**  
**Bandpass:** Lanczos lowpass at {LOWPASS_PERIOD} d, half-window {LOWPASS_HALF_W} d, edge drop ±{EDGE_DROP_DAYS} d/year. Result: {len(X_bp)} samples (down from {len(X)}).

## Three-way comparison (preliminary)

{df_comp.to_markdown(index=False)}

The conventional (PC1, PC2) leg is added by notebook 09.

## Month-clustering confound check

- Angle  ANOVA F = {f_a:.2f}  p = {p_a:.2e}
- Radius ANOVA F = {f_r:.2f}  p = {p_r:.2e}
- Threshold: F > 50 = strong confound, model learned "this is month X" instead of "this is BSISO phase Y".

## Decision

{decision_text}

## Files produced

```
checkpoints/encoder_{RUN_TAG}_final.pth
checkpoints/training_history_{RUN_TAG}.json
data/processed/X_MJJAS_lee_lp25.npy             ({len(X_bp)} samples, bandpassed)
data/processed/labels_aligned_mjjas_lee_lp25.csv  (rows aligned to bandpassed)
results/lee_2d_ssl/embeddings.npy
results/lee_2d_ssl/training_curves.png
results/lee_2d_ssl/embedding_2d_overview.png
results/lee_2d_ssl/month_clustering_check.png
results/lee_2d_ssl/linear_probe_results.json
results/lee_2d_ssl/enso_displacement.png
results/lee_2d_ssl/phase2_ssl_summary.md  (this file)
```
"""

with open(f'{RESULTS_DIR}/phase2_ssl_summary.md', 'w') as f:
    f.write(summary)
print(f'\nSaved: {RESULTS_DIR}/phase2_ssl_summary.md')

## Cell 15 — (Optional) Download All Outputs

In [ ]:
from google.colab import files
for fname in sorted(os.listdir(RESULTS_DIR)):
    files.download(f'{RESULTS_DIR}/{fname}')

---
## Done!

**Send back to me:**
1. `phase2_ssl_summary.md` — the headline result + auto-decision.
2. `embedding_2d_overview.png` — 4-panel scatter (the by-phase and by-month panels are the key reads).
3. `month_clustering_check.png` — the formal confound check.
4. `training_curves.png` — sanity that training was healthy.
5. `enso_displacement.png` — does ENSO modulation emerge without labels?

I'll log Phase 2 results into the conversation log + plan checklist, and either:
- start `09_threeway_comparison.ipynb` (Phase 3) with all three representations + lag probes, or
- iterate on the bandpass / pair definition if there's a month confound or weak signal.

---
*DDCS Project | jh9141@nyu.edu*